In [ ]:
# Mounting Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Pointing to dataset
import os
data_root = "/content/drive/MyDrive/Phytoplankton_Data/Library"
print("exists:", os.path.exists(data_root))
print("classes:", [d for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root,d))][:30])

exists: True
classes: ['Nanophytoplanton', 'Cyanobacteria', 'Prochlorococcus', 'Nitzschia', 'Mineral particles', 'Picocyanobacteria', 'Crocosphaera', 'Synechococcus']


In [ ]:
# Counting images per class

import glob, pandas as pd, os

rows=[]
for c in sorted(os.listdir(data_root)):
    p = os.path.join(data_root, c)
    if os.path.isdir(p):
        n = len(glob.glob(p+"/*.jpg")) + len(glob.glob(p+"/*.png")) + len(glob.glob(p+"/*.jpeg"))
        rows.append((c,n))

df = pd.DataFrame(rows, columns=["class","count"]).sort_values("count", ascending=False)
df

,class,count
5,Picocyanobacteria,4071
3,Nanophytoplanton,3503
6,Prochlorococcus,2210
7,Synechococcus,305
2,Mineral particles,239
0,Crocosphaera,66
4,Nitzschia,42
1,Cyanobacteria,37


In [ ]:
# ===== resume counts: N, K, S =====
import os, glob, json

# EDIT if your roots differ
ROOTS = [
    "/content/drive/MyDrive/Phytoplankton_Data/Library",
    "/content/drive/MyDrive/Phytoplankton_Data/Library-2",
]

IMG_EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

existing_roots = [r for r in ROOTS if os.path.isdir(r)]
print("Using roots:")
for r in existing_roots:
    print(" -", r)

# K = number of class folders (union across roots)
class_set = set()
# N = total images across all class folders (unique by path)
all_images = set()
# S = number of distinct roots actually used
S = len(existing_roots)

for root in existing_roots:
    for cls in os.listdir(root):
        cls_path = os.path.join(root, cls)
        if not os.path.isdir(cls_path):
            continue
        class_set.add(cls)
        for ext in IMG_EXTS:
            for p in glob.glob(os.path.join(cls_path, f"*{ext}")):
                all_images.add(p)

K = len(class_set)
N = len(all_images)

print("\n=== COUNTS ===")
print("N (total images):", N)
print("K (classes/species):", K)
print("S (sources/roots merged):", S)

print("\nClasses:", sorted(class_set))

Using roots:
 - /content/drive/MyDrive/Phytoplankton_Data/Library
 - /content/drive/MyDrive/Phytoplankton_Data/Library-2

=== COUNTS ===
N (total images): 12315
K (classes/species): 10
S (sources/roots merged): 2

Classes: ['Crocosphaera', 'Cyanobacteria', 'Mineral particles', 'Nanophytoplanton', 'Nitzschia', 'Pennate Diatoms', 'Picocyanobacteria', 'Prochlorococcus', 'Richelia', 'Synechococcus']


In [ ]:
# ===== optional: count clustered images you produced =====
import os, glob

OUTPUT_ROOTS = [
    "/content/drive/MyDrive/Phytoplankton_Outputs",
    "/content/drive/MyDrive/Phytoplankton_Deploy",
]

cluster_imgs = 0
cluster_folders = 0

for root in OUTPUT_ROOTS:
    if not os.path.isdir(root):
        continue
    for dirpath, dirnames, filenames in os.walk(root):
        base = os.path.basename(dirpath)
        if base.startswith("cluster_"):
            cluster_folders += 1
            for fn in filenames:
                if fn.lower().endswith((".jpg",".jpeg",".png",".tif",".tiff")):
                    cluster_imgs += 1

print("Cluster folders created:", cluster_folders)
print("Clustered images saved:", cluster_imgs)

Cluster folders created: 71
Clustered images saved: 1626


In [ ]:
print("min:", df["count"].min(), "max:", df["count"].max(), "classes:", len(df))

min: 37 max: 4071 classes: 8


In [ ]:
import tensorflow as tf
print(tf.__version__)

2.19.0


In [ ]:
import os, glob, numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# ===== FlowCAM Discrete — Step 1: Probe structure (run once, then set cols below) =====
import os, glob, pandas as pd

CAST_ROOT = "/content/drive/MyDrive/BLOOFINZ-FlowCAM-2022/Bloofin-2022-Discrete"

cast_dirs = sorted(glob.glob(os.path.join(CAST_ROOT, "BFZ-Cast*")))
print(f"Cast folders found: {len(cast_dirs)}")
for d in cast_dirs:
    print(" ", os.path.basename(d))

# Probe the first cast to reveal column names and image naming
first = cast_dirs[0]
csvs  = glob.glob(os.path.join(first, "*.csv"))
seg_dirs = glob.glob(os.path.join(first, "*_Segments"))

print(f"\n--- Probing: {os.path.basename(first)} ---")
print(f"CSVs found : {[os.path.basename(c) for c in csvs]}")

if seg_dirs:
    seg_samples = glob.glob(os.path.join(seg_dirs[0], "*"))[:6]
    print(f"_Segments/ : {[os.path.basename(s) for s in seg_samples]}")
    print(f"Extensions : {set(os.path.splitext(s)[1].lower() for s in glob.glob(os.path.join(seg_dirs[0], '*')))}")
else:
    print("_Segments/ : NOT FOUND — check folder name")

if csvs:
    probe = pd.read_csv(csvs[0], nrows=4)
    print(f"\nCSV columns ({len(probe.columns)}) with sample values:")
    for col in probe.columns:
        print(f"  {col!r:40s} → {probe[col].iloc[0]!r}")


In [ ]:
# ===== FlowCAM Discrete — Step 3: tf.data pipeline (drop-in replacement for Library loading) =====
#
# Produces the same variable names as the old library pipeline so every cell below
# (model build, training, evaluation, export) works unchanged:
#   class_names, class_to_id, train_files, val_files,
#   train_labels, val_labels, train_ds, val_ds, steps_per_epoch, class_weight
#
import numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split

IMG_SIZE   = (384, 384)
BATCH_SIZE = 32
SEED       = 42
VAL_SPLIT  = 0.2
AUTOTUNE   = tf.data.AUTOTUNE

# Drop classes with fewer than this many images (too few to stratify or learn from)
MIN_SAMPLES = 10

counts_per_class = manifest["label"].value_counts()
keep    = counts_per_class[counts_per_class >= MIN_SAMPLES].index
dropped = counts_per_class[counts_per_class <  MIN_SAMPLES]
if len(dropped):
    print(f"Dropping {len(dropped)} class(es) with < {MIN_SAMPLES} samples:")
    print(dropped.to_string(), "\n")

manifest_f  = manifest[manifest["label"].isin(keep)].reset_index(drop=True)
class_names = sorted(manifest_f["label"].unique().tolist())
class_to_id = {c: i for i, c in enumerate(class_names)}

filepaths = manifest_f["path"].values
labels    = manifest_f["label"].map(class_to_id).values.astype(np.int32)

print(f"Classes  ({len(class_names)}): {class_names}")
print(f"Images after filter: {len(filepaths):,}")

# --- stratified split ---
train_files, val_files, train_labels, val_labels = train_test_split(
    filepaths, labels,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=labels,
)
print(f"Train: {len(train_files):,}  |  Val: {len(val_files):,}")

# --- image decoder ---
# Uses PIL so TIFF, PNG, JPEG all work without conversion.
def _pil_decode(path_tensor):
    from PIL import Image
    img = Image.open(path_tensor.numpy().decode("utf-8")).convert("RGB")
    img = img.resize((IMG_SIZE[1], IMG_SIZE[0]), Image.BILINEAR)  # PIL: (W, H)
    return np.array(img, dtype=np.float32)

def decode_resize(path, label):
    img = tf.py_function(_pil_decode, [path], tf.float32)
    img.set_shape(IMG_SIZE + (3,))
    return img, tf.cast(label, tf.int32)

# --- val dataset (unbalanced — mirrors real-world distribution) ---
val_ds = (
    tf.data.Dataset.from_tensor_slices((val_files, val_labels))
    .map(decode_resize, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# --- balanced train dataset (same sqrt-inverse-freq strategy as before) ---
train_files_by_class = {i: train_files[train_labels == i] for i in range(len(class_names))}

per_class_datasets = []
for i in range(len(class_names)):
    ds_i = tf.data.Dataset.from_tensor_slices(
        (train_files_by_class[i],
         np.full(len(train_files_by_class[i]), i, dtype=np.int32))
    )
    ds_i = ds_i.shuffle(
        min(len(train_files_by_class[i]), 5000),
        seed=SEED, reshuffle_each_iteration=True
    ).repeat()
    per_class_datasets.append(ds_i)

counts_arr  = np.bincount(train_labels, minlength=len(class_names)).astype(np.float32)
weights_inv = 1.0 / np.sqrt(np.maximum(counts_arr, 1.0))
mix         = 0.25
p_nat       = counts_arr / counts_arr.sum()
p_bal       = weights_inv / weights_inv.sum()
sample_weights = (mix * p_bal + (1.0 - mix) * p_nat).tolist()

train_ds = (
    tf.data.Dataset.sample_from_datasets(per_class_datasets, weights=sample_weights, seed=SEED)
    .map(decode_resize, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

steps_per_epoch = int(np.ceil(len(train_files) / BATCH_SIZE))
class_weight    = None   # balanced batches → no class_weight dict needed

print(f"\nsteps_per_epoch : {steps_per_epoch}")
print(f"val classes present: {len(np.unique(val_labels))} of {len(class_names)}")
print("\nTrain samples per class:")
for i, c in enumerate(class_names):
    print(f"  {c:<30s} {int(counts_arr[i]):>5d}")


In [ ]:
import os, numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split

IMG_SIZE = (384, 384)
BATCH_SIZE = 32
SEED = 42
VAL_SPLIT = 0.2
AUTOTUNE = tf.data.AUTOTUNE

data_root = "/content/drive/MyDrive/Phytoplankton_Data/Library"

# --- collect files + labels (folder name = class) ---
class_names = sorted([d for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d))])
class_to_id = {c:i for i,c in enumerate(class_names)}

filepaths, labels = [], []
for c in class_names:
    folder = os.path.join(data_root, c)
    for fn in os.listdir(folder):
        if fn.lower().endswith((".jpg", ".jpeg", ".png")):
            filepaths.append(os.path.join(folder, fn))
            labels.append(class_to_id[c])

filepaths = np.array(filepaths)
labels = np.array(labels)

print("num_classes:", len(class_names))
print("total images:", len(filepaths))

# --- stratified split ---
train_files, val_files, train_labels, val_labels = train_test_split(
    filepaths, labels,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=labels
)

def decode_resize(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)   # stable + faster than decode_image
    img = tf.image.resize(img, IMG_SIZE, method="bilinear")
    img = tf.cast(img, tf.float32)
    return img, tf.cast(label, tf.int32)

# --- normal val dataset ---
val_ds = tf.data.Dataset.from_tensor_slices((val_files, val_labels))
val_ds = val_ds.map(decode_resize, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# --- BALANCED train dataset (uniform-ish sampling across classes) ---
train_files_by_class = {i: train_files[train_labels == i] for i in range(len(class_names))}

per_class_datasets = []
for i in range(len(class_names)):
    ds_i = tf.data.Dataset.from_tensor_slices(
        (train_files_by_class[i], np.full(len(train_files_by_class[i]), i, dtype=np.int32))
    )
    ds_i = ds_i.shuffle(min(len(train_files_by_class[i]), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds_i = ds_i.repeat()  # oversample minority classes automatically
    per_class_datasets.append(ds_i)

counts = np.bincount(train_labels, minlength=len(class_names)).astype(np.float32)

# sqrt inverse freq = soft balancing (usually best ROI for val accuracy)
weights = 1.0 / np.sqrt(np.maximum(counts, 1.0))

# optional: mix with natural distribution so it doesn’t over-correct
mix = 0.25
p_nat = counts / counts.sum()
p_bal = weights / weights.sum()
weights = (mix * p_bal + (1.0 - mix) * p_nat).tolist()

train_ds = tf.data.Dataset.sample_from_datasets(
    per_class_datasets,
    weights=weights,
    seed=SEED
)

train_ds = train_ds.map(decode_resize, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

steps_per_epoch = int(np.ceil(len(train_files) / BATCH_SIZE))

print("train size:", len(train_files), "val size:", len(val_files))
print("steps_per_epoch:", steps_per_epoch)
print("classes:", class_names)
print("val classes present:", len(np.unique(val_labels)), "of", len(class_names))

num_classes: 8
total images: 10473
train size: 8378 val size: 2095
steps_per_epoch: 262
classes: ['Crocosphaera', 'Cyanobacteria', 'Mineral particles', 'Nanophytoplanton', 'Nitzschia', 'Picocyanobacteria', 'Prochlorococcus', 'Synechococcus']
val classes present: 8 of 8


In [ ]:
# We are using BALANCED batches, so class_weight is not needed (and often hurts stability).
class_weight = None

In [ ]:
# ### Augmentation + performance
# from tensorflow import keras
# from tensorflow.keras import layers
# AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
# safety: ensure num_classes exists
try:
    num_classes
except NameError:
    num_classes = len(class_names)

print("num_classes:", num_classes)

num_classes: 8


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# ---------- 0) speed / stability ----------
tf.keras.mixed_precision.set_global_policy("mixed_float16")  # A100 loves this

# ---------- 1) augmentation (don’t overdo contrast on blurry microscopy) ----------
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.06, 0.06),
], name="augment")

# ---------- 2) class prior (from TRAIN labels only) ----------
counts = np.bincount(train_labels, minlength=len(class_names)).astype(np.float32)
prior = counts / np.sum(counts)
log_prior = tf.constant(np.log(prior + 1e-12), dtype=tf.float32)  # shape (C,)

# ---------- 3) Balanced Softmax / Logit-adjusted CE (stable on imbalance + noise) ----------
class BalancedSoftmaxLoss(keras.losses.Loss):
    def __init__(self, log_prior, name="balanced_softmax"):
        super().__init__(name=name)
        self.log_prior = log_prior

    def call(self, y_true, y_pred_logits):
        # y_pred_logits: (B, C) logits
        y_true = tf.cast(y_true, tf.int32)
        # add log prior to logits (logit adjustment)
        adj_logits = tf.cast(y_pred_logits, tf.float32) + self.log_prior
        loss = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y_true, logits=adj_logits)
        return tf.reduce_mean(loss)

# ---------- 4) backbone ----------
inputs = keras.Input(shape=IMG_SIZE + (3,))
x = augment(inputs, training=True)
x = keras.applications.efficientnet_v2.preprocess_input(x)

base = keras.applications.EfficientNetV2L(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,)
)
base.trainable = False

x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(512, activation="swish")(x)
x = layers.Dropout(0.3)(x)

# IMPORTANT: output LOGITS (no softmax here)
logits = layers.Dense(len(class_names), dtype="float32")(x)  # force fp32 output for stability
model = keras.Model(inputs, logits)

# ---------- 5) optimizer + schedule ----------
steps_per_epoch = int(np.ceil(len(train_files) / BATCH_SIZE))
total_steps = steps_per_epoch * 10  # warmup phase epochs

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=2e-4,
    decay_steps=total_steps,
    alpha=0.1
)

opt = keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4)

loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(
    optimizer=opt,
    loss=loss_fn,
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="acc"),
        keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),
]
)

model.summary()

473176280/473176280 ━━━━━━━━━━━━━━━━━━━━ 12s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augment (Sequential)            │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-l (Functional)   │ (None, 12, 12, 1280)   │   117,746,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         4,104 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 118,411,944 (451.71 MB)

 Trainable params: 662,536 (2.53 MB)

 Non-trainable params: 117,749,408 (449.18 MB)

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("best.keras", monitor="val_acc", mode="max", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_acc", mode="max", patience=6, restore_best_weights=True),
]

print("=== Stage 1: head training (backbone frozen) ===")
hist1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    steps_per_epoch=steps_per_epoch,
    callbacks=callbacks
)

# print("=== Stage 2: fine-tune top of backbone (tiny LR) ===")
# base.trainable = True

# # Freeze BatchNorms (super important for small/noisy microscopy)
# for layer in base.layers:
#     if isinstance(layer, layers.BatchNormalization):
#         layer.trainable = False

# # unfreeze only last ~25% of layers (safe + effective)
# cut = int(len(base.layers) * 0.75)
# for layer in base.layers[:cut]:
#     layer.trainable = False

# # set a small constant LR for fine-tuning (NO schedule here)
# model.compile(
#     optimizer=keras.optimizers.AdamW(learning_rate=2e-5, weight_decay=1e-4),
#     loss=model.loss,
#     metrics=[
#         keras.metrics.SparseCategoricalAccuracy(name="acc"),
#         keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),
#     ],
# )

# hist2 = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=30,
#     steps_per_epoch=steps_per_epoch,
#     callbacks=callbacks
# )

=== Stage 1: head training (backbone frozen) ===
Epoch 1/12
262/262 ━━━━━━━━━━━━━━━━━━━━ 63s 241ms/step - acc: 0.5803 - loss: 1.0381 - top3: 0.9428 - val_acc: 0.5866 - val_loss: 0.9313 - val_top3: 0.9809
Epoch 2/12
262/262 ━━━━━━━━━━━━━━━━━━━━ 58s 221ms/step - acc: 0.5751 - loss: 1.0546 - top3: 0.9392 - val_acc: 0.5866 - val_loss: 0.9364 - val_top3: 0.9819
Epoch 3/12
262/262 ━━━━━━━━━━━━━━━━━━━━ 62s 235ms/step - acc: 0.5728 - loss: 1.0116 - top3: 0.9469 - val_acc: 0.5905 - val_loss: 0.9293 - val_top3: 0.9823
Epoch 4/12
262/262 ━━━━━━━━━━━━━━━━━━━━ 58s 220ms/step - acc: 0.5862 - loss: 1.0113 - top3: 0.9407 - val_acc: 0.5871 - val_loss: 0.9322 - val_top3: 0.9814
Epoch 5/12
262/262 ━━━━━━━━━━━━━━━━━━━━ 62s 237ms/step - acc: 0.5816 - loss: 1.0130 - top3: 0.9500 - val_acc: 0.5909 - val_loss: 0.9288 - val_top3: 0.9809
Epoch 6/12
262/262 ━━━━━━━━━━━━━━━━━━━━ 57s 219ms/step - acc: 0.6018 - loss: 0.9755 - top3: 0.9499 - val_acc: 0.5881 - val_loss: 0.9395 - val_top3: 0.9790
Epoch 7/12
262/262 ━━

In [ ]:
for x,y in train_ds.take(1):
    print("y dtype:", y.dtype, "y shape:", y.shape)

y dtype: <dtype: 'int32'> y shape: (32,)


In [ ]:
print("=== Stage 2: gentle fine-tune (very small unfreeze) ===")
base.trainable = True

# keep ALL BatchNorm frozen
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# unfreeze only last ~10% layers
cut = int(len(base.layers) * 0.95)
for layer in base.layers[:cut]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=5e-6, weight_decay=1e-4),
    loss=model.loss,
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="acc"),
],
)

hist2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    steps_per_epoch=steps_per_epoch,
    callbacks=callbacks
)

=== Stage 2: gentle fine-tune (very small unfreeze) ===
Epoch 1/10
262/262 ━━━━━━━━━━━━━━━━━━━━ 134s 289ms/step - acc: 0.6051 - loss: 0.9609 - val_acc: 0.5943 - val_loss: 0.9228
Epoch 2/10
262/262 ━━━━━━━━━━━━━━━━━━━━ 60s 230ms/step - acc: 0.6054 - loss: 0.9490 - val_acc: 0.5933 - val_loss: 0.9215
Epoch 3/10
262/262 ━━━━━━━━━━━━━━━━━━━━ 60s 230ms/step - acc: 0.6209 - loss: 0.9200 - val_acc: 0.5924 - val_loss: 0.9185
Epoch 4/10
262/262 ━━━━━━━━━━━━━━━━━━━━ 66s 254ms/step - acc: 0.6075 - loss: 0.9445 - val_acc: 0.6000 - val_loss: 0.9031
Epoch 5/10
262/262 ━━━━━━━━━━━━━━━━━━━━ 61s 233ms/step - acc: 0.6146 - loss: 0.9160 - val_acc: 0.6000 - val_loss: 0.9024
Epoch 6/10
262/262 ━━━━━━━━━━━━━━━━━━━━ 66s 251ms/step - acc: 0.6306 - loss: 0.8881 - val_acc: 0.6043 - val_loss: 0.9110
Epoch 7/10
262/262 ━━━━━━━━━━━━━━━━━━━━ 67s 254ms/step - acc: 0.6338 - loss: 0.8924 - val_acc: 0.6062 - val_loss: 0.8994
Epoch 8/10
262/262 ━━━━━━━━━━━━━━━━━━━━ 61s 235ms/step - acc: 0.6277 - loss: 0.8876 - val_acc: 0

In [ ]:
# stage1_w = "/content/stage1.weights.h5"
# model.save_weights(stage1_w)
# print("saved:", stage1_w)

In [ ]:
### Evaluation - Accuracy + Confusion Matrix + Macro F1

from tensorflow import keras
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

model = keras.models.load_model("best.keras", compile=False)

y_true, y_pred = [], []
for images, labels in val_ds:
    logits = model.predict(images, verbose=0)
    probs = tf.nn.softmax(logits, axis=1).numpy()
    preds = np.argmax(probs, axis=1)
    y_pred.extend(np.argmax(probs, axis=1).tolist())
    y_true.extend(labels.numpy().tolist())

print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))
print(confusion_matrix(y_true, y_pred))

                   precision    recall  f1-score   support

     Crocosphaera     0.0556    0.1538    0.0816        13
    Cyanobacteria     0.1304    0.4286    0.2000         7
Mineral particles     0.9583    0.9583    0.9583        48
 Nanophytoplanton     0.7474    0.6077    0.6703       701
        Nitzschia     0.6364    0.7778    0.7000         9
Picocyanobacteria     0.5745    0.7912    0.6656       814
  Prochlorococcus     0.4980    0.2783    0.3570       442
    Synechococcus     0.4872    0.3115    0.3800        61

         accuracy                         0.6062      2095
        macro avg     0.5110    0.5384    0.5016      2095
     weighted avg     0.6180    0.6062    0.5955      2095

[[  2   3   0   4   0   2   0   2]
 [  2   3   2   0   0   0   0   0]
 [  0   1  46   0   0   0   0   1]
 [ 21  10   0 426   3 187  37  17]
 [  0   2   0   0   7   0   0   0]
 [  3   0   0  82   0 644  85   0]
 [  1   0   0  33   0 285 123   0]
 [  7   4   0  25   1   3   2  19]]


In [ ]:
import os, json
import tensorflow as tf
from tensorflow import keras

DEPLOY_DIR = "/content/drive/MyDrive/Phytoplankton_Deploy/V9"
os.makedirs(DEPLOY_DIR, exist_ok=True)

# 1) Save class names (must match training order!)
# If you already have class_names, keep it.
# Otherwise manually set it here in the exact order you trained with.
with open(os.path.join(DEPLOY_DIR, "class_names.json"), "w") as f:
    json.dump(class_names, f)

# 2) Create an inference-only model that outputs probabilities
inp = model.input
out = model.output

# If your model outputs logits, force softmax.
# (Even if it's already probs, applying softmax again is usually harmless but we avoid double-softmax.)
# We'll detect by checking if values look like logits later in the app; here we just wrap safely.
probs = keras.layers.Softmax(axis=-1)(out)
infer_model = keras.Model(inp, probs, name="v9_infer")
infer_model.save(os.path.join(DEPLOY_DIR, "v9_infer.keras"))

print("Saved deploy artifacts to:", DEPLOY_DIR)
print("Files:", os.listdir(DEPLOY_DIR))

Saved deploy artifacts to: /content/drive/MyDrive/Phytoplankton_Deploy/V9
Files: ['class_names.json', 'v9_infer.keras']


In [ ]:
# ===== Export metrics.json → Drive + GitHub auto-commit =====
#
# Requires a GitHub personal access token with 'contents: write' scope.
# Store it in Colab Secrets (🔑 icon in the left sidebar) as:  GITHUB_TOKEN
# Never paste the token directly into the notebook.
#
import json, base64, datetime, os, requests, numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras
import tensorflow as tf

# --- config (edit if repo / branch / path differ) ---
DEPLOY_DIR = "/content/drive/MyDrive/Phytoplankton_Deploy/V9"
GH_REPO    = "adityaram-2003/phytoplankton-classification"
GH_PATH    = "metrics.json"   # path inside repo root
GH_BRANCH  = "main"

# Resolve token from Colab Secrets, then env var, then raise
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GH_TOKEN = os.environ.get("GITHUB_TOKEN", "")

if not GH_TOKEN:
    raise RuntimeError(
        "GITHUB_TOKEN not found.\n"
        "Add it via: Colab left sidebar → 🔑 Secrets → New secret → name=GITHUB_TOKEN"
    )

# --- 1. Run inference on val set using best saved model ---
model_eval = keras.models.load_model("best.keras", compile=False)

y_true, y_pred = [], []
for images, batch_labels in val_ds:
    logits = model_eval(images, training=False)
    probs  = tf.nn.softmax(logits, axis=1).numpy()
    y_pred.extend(np.argmax(probs, axis=1).tolist())
    y_true.extend(batch_labels.numpy().tolist())

# --- 2. Compute metrics ---
report = classification_report(
    y_true, y_pred,
    target_names=class_names,
    digits=4, output_dict=True, zero_division=0,
)
cm = confusion_matrix(y_true, y_pred).tolist()

# Combine Stage 1 + Stage 2 histories (hist2 is optional — skipped if not run)
def _floatify(h):
    return {k: [float(v) for v in vals] for k, vals in h.history.items()}

combined_history = _floatify(hist1)
try:
    for k, vals in _floatify(hist2).items():
        combined_history[k] = combined_history.get(k, []) + vals
except NameError:
    pass  # Stage 2 not executed in this session

metrics = {
    "timestamp":           datetime.datetime.utcnow().isoformat() + "Z",
    "model_version":       "V9",
    "dataset":             "BLOOFINZ-FlowCAM-2022 Discrete",
    "num_classes":         len(class_names),
    "class_names":         class_names,
    "val_accuracy":        round(report["accuracy"], 6),
    "macro_f1":            round(report["macro avg"]["f1-score"], 6),
    "weighted_f1":         round(report["weighted avg"]["f1-score"], 6),
    "per_class_f1":        {c: round(report[c]["f1-score"],    4) for c in class_names},
    "per_class_precision": {c: round(report[c]["precision"],   4) for c in class_names},
    "per_class_recall":    {c: round(report[c]["recall"],      4) for c in class_names},
    "per_class_support":   {c: int(report[c]["support"])          for c in class_names},
    "confusion_matrix":    cm,
    "training_history":    combined_history,
}

metrics_json = json.dumps(metrics, indent=2)

# --- 3. Save to Drive ---
os.makedirs(DEPLOY_DIR, exist_ok=True)
drive_out = os.path.join(DEPLOY_DIR, "metrics.json")
with open(drive_out, "w") as f:
    f.write(metrics_json)
print(f"Saved to Drive : {drive_out}")

# --- 4. Commit to GitHub via Contents API ---
api_url = f"https://api.github.com/repos/{GH_REPO}/contents/{GH_PATH}"
headers = {
    "Authorization": f"token {GH_TOKEN}",
    "Accept":        "application/vnd.github.v3+json",
}

# Fetch current file SHA — required by the API when updating an existing file
sha = None
get_resp = requests.get(api_url, headers=headers, params={"ref": GH_BRANCH})
if get_resp.status_code == 200:
    sha = get_resp.json()["sha"]        # file exists → update
elif get_resp.status_code == 404:
    sha = None                          # file does not exist yet → create
else:
    raise RuntimeError(f"GitHub GET {get_resp.status_code}: {get_resp.text}")

commit_msg = (
    f"chore: update metrics.json — "
    f"val_acc={metrics['val_accuracy']:.4f} "
    f"macro_f1={metrics['macro_f1']:.4f} "
    f"[{metrics['timestamp']}]"
)
payload = {
    "message": commit_msg,
    "content": base64.b64encode(metrics_json.encode()).decode(),
    "branch":  GH_BRANCH,
}
if sha:
    payload["sha"] = sha   # must be included when updating; omit on first create

put_resp = requests.put(api_url, headers=headers, json=payload)
if put_resp.status_code in (200, 201):
    commit_url = put_resp.json()["commit"]["html_url"]
    print(f"GitHub commit  : {commit_url}")
else:
    raise RuntimeError(f"GitHub PUT {put_resp.status_code}: {put_resp.text}")

# --- 5. Summary ---
print(f"\nval_accuracy : {metrics['val_accuracy']:.4f}")
print(f"macro_f1     : {metrics['macro_f1']:.4f}")
print(f"weighted_f1  : {metrics['weighted_f1']:.4f}")
print("\nper_class_f1:")
for c, f1 in metrics["per_class_f1"].items():
    bar = "█" * int(f1 * 20)
    print(f"  {c:<30s} {f1:.4f}  {bar}")


In [ ]:
import numpy as np, json, os
from sklearn.metrics import classification_report, confusion_matrix

DEPLOY_DIR = "/content/drive/MyDrive/Phytoplankton_Deploy/V9"

y_true, y_pred = [], []
for x, y in val_ds:
    p = infer_model.predict(x, verbose=0)
    y_pred.extend(np.argmax(p, axis=1).tolist())
    y_true.extend(y.numpy().tolist())

report = classification_report(
    y_true, y_pred, target_names=class_names,
    digits=4, output_dict=True, zero_division=0
)
cm = confusion_matrix(y_true, y_pred).tolist()

with open(os.path.join(DEPLOY_DIR, "val_metrics.json"), "w") as f:
    json.dump({"report": report, "confusion_matrix": cm}, f)

print("Saved val_metrics.json")
print("Val accuracy:", report["accuracy"])

Saved val_metrics.json
Val accuracy: 0.6062052505966588
